# 2. Representation and chemical space

Choosing a representation matters more than choosing a model. A random
forest on good features beats a tuned neural network on bad ones, and no
amount of hyperparameter search recovers information the representation
threw away.

**Covers:** `qsarkit.representation`, `qsarkit.transform`,
`qsarkit.chemspace`, `qsarkit.neighbors`, `qsarkit.cluster`

In [1]:
import numpy as np
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")
np.set_printoptions(precision=3, suppress=True)

SMILES = [
    "OC(=O)c1ccccc1", "OC(=O)c1ccc(C)cc1", "OC(=O)c1ccc(Cl)cc1",
    "OC(=O)c1ccc(Br)cc1", "OC(=O)c1ccc(OC)cc1", "OC(=O)c1ccc(N)cc1",
    "CC(=O)Nc1ccccc1", "CC(=O)Nc1ccc(C)cc1", "CC(=O)Nc1ccc(Cl)cc1",
    "CC(=O)Nc1ccc(F)cc1", "CC(=O)Nc1ccc(OC)cc1", "CC(=O)Nc1ccc(O)cc1",
    "NC(=O)c1ccncc1", "NC(=O)c1ccc(C)nc1", "NC(=O)c1ccc(Cl)nc1",
    "NC(=O)c1ccc(OC)nc1", "CNC(=O)c1ccncc1", "CCNC(=O)c1ccncc1",
    "c1ccc2[nH]cnc2c1", "Cc1ccc2[nH]cnc2c1", "Clc1ccc2[nH]cnc2c1",
    "COc1ccc2[nH]cnc2c1", "Cn1cnc2ccccc21", "CCn1cnc2ccccc21",
]
Y = np.array([
    5.10, 5.35, 7.80, 5.40, 5.05, 4.90,
    6.20, 6.45, 6.70, 6.55, 6.10, 6.05,
    7.10, 7.35, 7.55, 7.20, 7.05, 6.95,
    8.00, 8.25, 8.45, 8.10, 7.90, 7.85,
])
mols = [Chem.MolFromSmiles(s) for s in SMILES]
FAMILY = (["benzoic acid"] * 6 + ["anilide"] * 6
          + ["pyridine amide"] * 6 + ["benzimidazole"] * 6)
len(mols)

24

## Fingerprints

Morgan (ECFP) fingerprints encode circular atom environments. They are the
default for good reason: cheap, effective with tree ensembles, and the
natural input to Tanimoto-based methods.

The radius is the substantive choice. Radius 2 (ECFP4) captures functional
groups; radius 3 (ECFP6) captures larger motifs at the cost of sparsity.

In [2]:
from qsarkit.representation import MorganFingerprint

X = MorganFingerprint(radius=2, n_bits=1024).transform(mols)
print("shape:", X.shape)
print("bits that ever fire:", int((X.sum(axis=0) > 0).sum()))
print("mean bits set per molecule:", X.sum(axis=1).mean())

shape: (24, 1024)
bits that ever fire: 133
mean bits set per molecule: 20.458333333333332


Fingerprints are sparse: most bits never fire on a small dataset. That is
not waste — it is what makes the representation general — but it does mean
a variance filter is nearly free (notebook 3).

In [3]:
from qsarkit.representation import (
    AtomPairFingerprint, MACCSKeysFingerprint, RDKitFingerprint,
    TopologicalTorsionFingerprint)

for name, fp in [
    ("Morgan r=2", MorganFingerprint(n_bits=512)),
    ("MACCS keys", MACCSKeysFingerprint()),
    ("RDKit path", RDKitFingerprint(n_bits=512)),
    ("Atom pair", AtomPairFingerprint(n_bits=512)),
    ("Torsion", TopologicalTorsionFingerprint(n_bits=512)),
]:
    Xi = fp.transform(mols)
    print(f"{name:12} shape={str(Xi.shape):10} "
          f"active columns={int((Xi.sum(axis=0) > 0).sum()):4}")

Morgan r=2   shape=(24, 512)  active columns= 124
MACCS keys   shape=(24, 167)  active columns=  59
RDKit path   shape=(24, 512)  active columns= 491
Atom pair    shape=(24, 512)  active columns= 149
Torsion      shape=(24, 512)  active columns=  62


MACCS keys are 166 hand-curated substructure questions: interpretable and
fixed-length, but far less expressive than a hashed fingerprint. Atom pairs
and topological torsions are **count** vectors by design — they encode how
often a feature occurs, not merely whether it does.

## Descriptors

Where fingerprints answer "what substructures are present", descriptors
answer "what is this molecule like".

In [4]:
from qsarkit.representation import PhysicochemicalDescriptors

block = PhysicochemicalDescriptors()
D = block.transform(mols)
import pandas as pd
frame = pd.DataFrame(D, columns=block.get_feature_names_out())
frame.insert(0, "family", FAMILY)
frame.head()

,family,MolWt,MolLogP,TPSA,NumHDonors,NumHAcceptors,NumRotatableBonds,FractionCSP3,MolMR,QED
0,benzoic acid,122.123,1.38480,37.30,1.0,1.0,1.0,0.000,33.4013,0.610604
1,benzoic acid,136.150,1.69322,37.30,1.0,1.0,1.0,0.125,38.1383,0.637460
2,benzoic acid,156.568,2.03820,37.30,1.0,1.0,1.0,0.000,38.4113,0.675780
3,benzoic acid,201.019,2.14730,37.30,1.0,1.0,1.0,0.000,41.1013,0.755096
4,benzoic acid,152.149,1.39340,46.53,1.0,2.0,2.0,0.125,39.9533,0.696129


Note the scales: molecular weight in the hundreds, logP in single digits.
Any distance-based or regularized model needs these scaled first —
fingerprints, being binary, do not.

In [5]:
print(frame.drop(columns="family").describe().loc[["mean", "std"]].round(2))

       MolWt  MolLogP   TPSA  NumHDonors  NumHAcceptors  NumRotatableBonds  \
mean  146.31     1.42  39.16        1.00           1.58               1.00   
std    17.68     0.64  13.10        0.42           0.58               0.59   

      FractionCSP3  MolMR   QED  
mean          0.10  40.22  0.63  
std           0.08   3.65  0.05  


In [6]:
from qsarkit.transform import DescriptorScaler

scaled = DescriptorScaler(method="standard").fit_transform(D)
print("after scaling — mean:", scaled.mean().round(6), " std:", scaled.std().round(3))

after scaling — mean: -0.0  std: 1.0


## Combining representations

Fingerprints and descriptors capture different things, and combining them
needs no new machinery.

In [7]:
from qsarkit.representation import FingerprintCombiner
from qsarkit.transform import MoleculeFeatureUnion

combined = FingerprintCombiner([
    ("morgan", MorganFingerprint(n_bits=256)),
    ("maccs", MACCSKeysFingerprint()),
])
print("fingerprint combiner:", combined.transform(mols).shape)

union = MoleculeFeatureUnion([
    ("maccs", MACCSKeysFingerprint()),
    ("physchem", PhysicochemicalDescriptors()),
])
print("feature union:      ", union.fit_transform(mols).shape)

fingerprint combiner: (24, 423)
feature union:       (24, 176)


## The right distance metric

Euclidean distance is wrong for sparse binary fingerprints: two molecules
that share no substructures are "close" in Euclidean terms because they
agree on the thousands of bits that are jointly zero — bits carrying no
chemical information.

Here is that failure, concretely.

In [8]:
from scipy.spatial.distance import euclidean
from qsarkit.neighbors import jaccard_distance

# Two synthetic pairs, constructed to differ in exactly the same number of
# bits — so Euclidean distance cannot tell them apart at all.
n = 1024
big_a = np.zeros(n); big_a[:40] = 1
big_b = np.zeros(n); big_b[10:50] = 1     # 30 bits shared out of 50

small_a = np.zeros(n); small_a[:12] = 1
small_b = np.zeros(n); small_b[10:22] = 1  # 2 bits shared out of 22

for label, (u, v) in [("similar, feature-rich", (big_a, big_b)),
                      ("dissimilar, sparse", (small_a, small_b))]:
    print(f"{label:22} euclidean={euclidean(u, v):.3f}  "
          f"tanimoto={1 - jaccard_distance(u, v):.3f}")

similar, feature-rich  euclidean=4.472  tanimoto=0.600
dissimilar, sparse     euclidean=4.472  tanimoto=0.091


Identical Euclidean distance; a 6.6-fold difference in Tanimoto similarity.

Euclidean distance on binary vectors is just the square root of the number
of differing bits — it never looks at how many bits the two molecules share
relative to how many they set between them. So it confuses "two elaborate
molecules with most of their features in common" with "two sparse molecules
with almost nothing in common".

On real fingerprints the effect shows up as a systematic bias toward calling
small molecules similar to everything.

In [9]:
Xb = MorganFingerprint(n_bits=1024).transform(mols)

print("real molecules, same comparison:")
for label, (i, j) in [("benzoic acid vs its 4-Me analogue", (0, 1)),
                      ("benzoic acid vs benzimidazole", (0, 18))]:
    print(f"  {label:36} euclidean={euclidean(Xb[i], Xb[j]):.3f}  "
          f"tanimoto={1 - jaccard_distance(Xb[i], Xb[j]):.3f}")

real molecules, same comparison:
  benzoic acid vs its 4-Me analogue    euclidean=3.000  tanimoto=0.550
  benzoic acid vs benzimidazole        euclidean=4.796  tanimoto=0.179


Every neighbour and clustering method in qsarkit uses Tanimoto/Jaccard,
verified against RDKit's `BulkTanimotoSimilarity`.

In [10]:
from qsarkit.neighbors import JaccardNeighborSearch

search = JaccardNeighborSearch(n_neighbors=4).fit(Xb)
distances, indices = search.kneighbors(Xb[:1])
print("nearest neighbours of", SMILES[0])
for dist, idx in zip(distances[0], indices[0]):
    print(f"  {1 - dist:.3f}  {SMILES[idx]:24} ({FAMILY[idx]})")

nearest neighbours of OC(=O)c1ccccc1
  1.000  OC(=O)c1ccccc1           (benzoic acid)
  0.550  OC(=O)c1ccc(N)cc1        (benzoic acid)
  0.550  OC(=O)c1ccc(C)cc1        (benzoic acid)
  0.550  OC(=O)c1ccc(Br)cc1       (benzoic acid)


## Clustering

Taylor–Butina is the standard cheminformatics clustering: no `n_clusters`
to choose, just a similarity cutoff, and every cluster has a real molecule
at its centre rather than an average corresponding to nothing.

In [11]:
from qsarkit.cluster import ButinaClustering

labels = ButinaClustering(cutoff=0.6).fit_predict(Xb)
frame = pd.DataFrame({"family": FAMILY, "cluster": labels})
print(pd.crosstab(frame["family"], frame["cluster"]))

cluster         0  1  2  3  4  5
family                          
anilide         1  5  0  0  0  0
benzimidazole   0  0  0  4  2  0
benzoic acid    6  0  0  0  0  0
pyridine amide  0  0  5  0  0  1


The clusters recover the substituent families without being told they
exist — which is the check worth doing before trusting a clustering.

In [12]:
from qsarkit.cluster import MaxMinPicker

picks = MaxMinPicker(n_to_pick=6, seed_index=0).fit(Xb).picks_
print("a maximally diverse subset of 6:")
for i in picks:
    print(f"  {SMILES[i]:24} ({FAMILY[i]})")

a maximally diverse subset of 6:
  OC(=O)c1ccccc1           (benzoic acid)
  COc1ccc2[nH]cnc2c1       (benzimidazole)
  Cn1cnc2ccccc21           (benzimidazole)
  NC(=O)c1ccc(OC)nc1       (pyridine amide)
  CC(=O)Nc1ccc(Cl)cc1      (anilide)
  CCNC(=O)c1ccncc1         (pyridine amide)


Each pick is the compound furthest from everything already picked, so the
selection spans the space rather than clustering in its densest region —
which is what a random sample would do.

## Chemical space

The questions to ask *before* modelling: does this library cover one region
or several, does the test set sit inside the training set's cloud.

In [13]:
from qsarkit.chemspace import ChemicalSpaceAnalyzer

analyzer = ChemicalSpaceAnalyzer(method="pca", random_state=0).fit(mols)
figure = analyzer.plot(labels=FAMILY, title="Chemical space (PCA on ECFP4)")
figure

> **On t-SNE and UMAP.** They preserve local neighbourhoods and give the
> familiar island plots, but between-cluster distances in those plots are
> **not** meaningful. Reading them as chemical distance is the commonest
> misuse of the technique. PCA's axes are interpretable and its distances
> mean something; prefer it unless you specifically need the local
> structure.

In [14]:
from qsarkit.chemspace import DiversityAnalyzer

report = DiversityAnalyzer().analyze(mols)
for key in ("n_molecules", "internal_diversity", "n_scaffolds",
            "scaffold_diversity", "acyclic_fraction"):
    print(f"{key:22} {report[key]:.3f}")

n_molecules            24.000
internal_diversity     0.763
n_scaffolds            3.000
scaffold_diversity     0.125
acyclic_fraction       0.000


In [15]:
comparison = DiversityAnalyzer().compare({
    "benzoic acids": mols[:6],
    "anilides": mols[6:12],
    "pyridine amides": mols[12:18],
    "benzimidazoles": mols[18:],
})
comparison[["n_molecules", "internal_diversity", "n_scaffolds"]].round(3)

,n_molecules,internal_diversity,n_scaffolds
0,6.0,0.471,1.0
1,6.0,0.388,1.0
2,6.0,0.611,1.0
3,6.0,0.641,1.0


## Scaffolds

Scaffolds are how medicinal chemists actually partition a library, and the
scaffold distribution says something a compound count cannot.

In [16]:
from qsarkit.chemspace import ScaffoldAnalyzer

scaffolds = ScaffoldAnalyzer().fit(mols)
print("distinct scaffolds:", scaffolds.n_scaffolds)
for smi, count in scaffolds.most_common():
    print(f"  {count:2}x  {smi}")

distinct scaffolds: 3
  12x  c1ccccc1
   6x  c1ccncc1
   6x  c1ccc2[nH]cnc2c1


Only three scaffolds, not four: the benzoic acids and the anilides share a
benzene framework once their acyclic side chains are stripped. That is
exactly the behaviour a scaffold split relies on, and it is why the split in
notebook 3 is harder than it looks.

## Novelty and coverage

How different is one set from another? This is the same question an
applicability domain asks, in library terms.

In [17]:
from qsarkit.chemspace import ChemicalSpaceCoverage, NearestNeighborAnalyzer

nn = NearestNeighborAnalyzer().fit(mols[:18])
novelty = nn.novelty(mols[18:])
print("novelty of the benzimidazoles vs the other three families:")
print(" ", novelty.round(3))

coverage = ChemicalSpaceCoverage(threshold=0.5).compare(mols[18:], mols[:18])
print("\nnovel fraction:", round(coverage["novel_fraction"], 3))

novelty of the benzimidazoles vs the other three families:
  [0.821 0.788 0.788 0.722 0.824 0.829]

novel fraction: 1.0


Entirely novel — which is what makes holding them out a genuine test, and
what notebook 3 does next.